# eCAADe Workshop 2026: Explainable Urban-Tree Growth ML

This notebook is a step-by-step exercise. It fits **one pooled XGBoost model**: species and monitoring period are categorical predictors, but no separate species, conifer, or broadleaf models are trained.

The five stages are:

1. environment setting and data preparation;
2. leakage-safe processing and VIF diagnosis;
3. training, validation, 85% refitting, and locked testing;
4. **environment-only** SHAP explanation;
5. ONNX export and parity testing.

The response is y = ln(g), where g = [ln(C_end) - ln(C_start)] / years.

## 1. Environment setting and data preparation

### Cell purpose: install workshop-only dependencies

This cell installs XGBoost, SHAP, and the ONNX libraries that may be absent from a fresh Colab runtime. Colab already supplies NumPy, pandas, scikit-learn, and Matplotlib, so the cell does not replace those core packages.

**Output:** installation messages only. A successful cell prepares the runtime; it does not load or change the data.

In [ ]:
# Colab already provides NumPy, pandas, scikit-learn, and Matplotlib.
%pip install -q "xgboost>=2.0,<4" "shap>=0.44,<1" "joblib>=1.3,<2" "onnx>=1.16,<2" "onnxmltools>=1.12,<2" "onnxruntime>=1.17,<2" 

### Cell purpose: locate the teaching materials

This cell uses the current folder when the notebook is run from a local checkout. In Colab it clones the GitHub repository once and points to the workshop sample CSV.

**Output:** the resolved workshop and CSV paths. No modelling occurs here.

In [ ]:
from pathlib import Path
import subprocess, sys

REPOSITORY = "https://github.com/sleepyheadzzzzzz/Tree-Point-Cloud-Training-and-Analysing.git"
REPO_DIR = Path("/content/Tree-Point-Cloud-Training-and-Analysing")
LOCAL_CANDIDATE = Path.cwd()

if (LOCAL_CANDIDATE / "urban_tree_ml_workshop_2026.py").exists():
    WORKSHOP_DIR = LOCAL_CANDIDATE
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO_DIR)], check=True)
    WORKSHOP_DIR = REPO_DIR / "eCAADe_workshop_2026_material"

sys.path.insert(0, str(WORKSHOP_DIR))
INPUT_CSV = WORKSHOP_DIR / "data" / "tree_carbon_ml_teaching_sample.csv"
OUTPUT_DIR = Path("/content/eCAADe_2026_outputs") if Path("/content").exists() else WORKSHOP_DIR / "outputs"
print("Workshop folder:", WORKSHOP_DIR)
print("Teaching CSV:", INPUT_CSV)

### Cell purpose: import the workflow functions and check versions

This cell imports each stage as a named function. Keeping stages separate lets us inspect VIF and each SHAP view independently. The environment table records the versions needed for reproducibility.

**Output:** a package-version table.

In [ ]:
import pandas as pd
from urban_tree_ml_workshop_2026 import (
    WorkflowConfig,
    compute_environment_shap,
    environment_report,
    export_onnx_and_examples,
    load_tree_level_data,
    plot_environment_beeswarm,
    plot_environment_dependence,
    plot_environment_spatial,
    plot_environment_waterfall,
    prepare_data,
    run_vif_analysis,
    train_validate_refit_test,
    xgb_parameter_table,
)

display(pd.Series(environment_report(), name="version").to_frame())

### Cell purpose: inspect the untouched tree-level sample

This cell reads the source CSV, reports its size, previews three rows, and counts retained species labels. It is a data audit only: transformations are performed later on a copy.

**Output:** dataset dimensions, a short preview, and species counts.

In [ ]:
tree_level = load_tree_level_data(INPUT_CSV)
print(f"Tree-level sample: {len(tree_level):,} rows x {tree_level.shape[1]} columns")
display(tree_level.head(3))
display(
    tree_level.groupby(["Species", "Species_Name"])
    .size()
    .rename("tree_rows")
    .to_frame()
)

## 2. Data processing

### Cell purpose: split trees and construct period observations

This cell creates a fixed configuration, assigns complete trees to 70% training, 15% validation, and 15% test partitions, and only then converts each tree to as many as three monitoring-period rows. It computes log starting height, annualized specific growth, log-SGR, annual percentage growth, and annual kg C gain.

Assigning the split first prevents observations from the same tree appearing in different partitions.

**Output:** the processing audit, split sizes, and a preview of the prepared long table.

In [ ]:
config = WorkflowConfig(
    input_csv=INPUT_CSV,
    output_dir=OUTPUT_DIR,
    random_state=2026,
    shap_sample=1200,
    example_rows=12,
)

prepared = prepare_data(config)
display(prepared["processing_audit"])
display(prepared["split_summary"])
display(prepared["long"].head(5))

### Cell purpose: verify leakage control and response transformations

This cell checks that every tree identifier occurs in exactly one partition. It then summarizes the modelling target and its two back-transformed forms by split.

**Output:** a passed leakage assertion and descriptive response statistics. If the assertion fails, modelling must stop.

In [ ]:
split_counts_per_tree = (
    prepared["long"]
    .groupby("Original_Tree_RowID")["Split"]
    .nunique()
)
assert split_counts_per_tree.max() == 1
print("Leakage check passed. Maximum partitions per tree:", split_counts_per_tree.max())

display(
    prepared["long"]
    .groupby("Split")[
        [
            "Log_Annualized_Specific_Growth",
            "Observed_Annual_Growth_Percent",
            "Observed_Annual_Carbon_Gain_kg",
        ]
    ]
    .describe()
)

### Cell purpose: calculate variance inflation factors (VIF)

VIF asks how well each continuous predictor can be reconstructed as a linear combination of the other continuous predictors. The calculation uses **training rows only**. Rough teaching guides are VIF below 5 = low concern, 5–10 = review, and above 10 = high linear collinearity.

Categorical one-hot columns are excluded because keeping all category dummies creates exact dummy-variable dependence. VIF is not a nonlinear-dependence test and is not an automatic feature-removal rule for XGBoost.

**Output:** a VIF table and the training-only Pearson correlation matrix.

In [ ]:
vif_outputs = run_vif_analysis(prepared)
display(vif_outputs["vif"].style.format({
    "auxiliary_R2": "{:.3f}",
    "tolerance": "{:.3f}",
    "VIF": "{:.2f}",
}))
display(
    vif_outputs["correlation"]
    .style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1)
    .format("{:.2f}")
)

## 3. Model training, validation, and locked test

### Cell purpose: expose every important XGBoost setting

This cell displays the exact learning rate, tree depth, row and column subsampling, split penalty, regularization, estimator cap, and early-stopping rule before training. These values are fixed in the script so the notebook explanation and executed model cannot drift apart.

**Output:** a three-column settings table showing the value and its role.

In [ ]:
settings = xgb_parameter_table(config.random_state)
display(settings)

### Cell purpose: select boosting rounds, refit, and lock the test

The initial model is trained on 70% of trees. Validation RMSE selects the number of boosting rounds through early stopping. The preprocessor and XGBoost model are then refitted on the combined 85% development data with that selected tree count. Finally, the 15% test is predicted once.

The test set never chooses learning rate, depth, subsampling, regularization, or stopping time.

**Output:** the selected number of trees and an in-memory fitted model package.

In [ ]:
trained = train_validate_refit_test(prepared, config)
print("Validation-selected XGBoost trees:", trained["selected_trees"])
print("Engineered feature count:", trained["x_test"].shape[1])

### Cell purpose: compare validation and locked-test metrics

This cell reports R², RMSE, and MAE on log-SGR, then back-transformed errors in annual percentage points and kg C tree⁻¹ yr⁻¹. Only the log-SGR values are errors on the direct model output.

**Output:** one validation row and one locked-test row.

In [ ]:
display(trained["metrics"].style.format(precision=4))

### Cell purpose: visually inspect locked-test calibration

This diagnostic plots observed versus predicted annual percentage growth. The dashed 1:1 line represents perfect agreement. Axes are limited at the 99.5th percentile so rare extreme back-transformations do not hide the main point cloud.

**Output:** an observed-versus-predicted scatter plot. This plot is diagnostic; it does not change the model.

In [ ]:
import matplotlib.pyplot as plt
pred = pd.read_csv(OUTPUT_DIR / "tables" / "locked_test_predictions.csv")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    pred["Observed_Annual_Growth_Percent"],
    pred["Predicted_Annual_Growth_Percent"],
    s=8,
    alpha=0.25,
)
limit = pred[
    ["Observed_Annual_Growth_Percent", "Predicted_Annual_Growth_Percent"]
].quantile(0.995).max()
ax.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
ax.set(
    xlim=(0, limit),
    ylim=(0, limit),
    xlabel="Observed annual growth (%)",
    ylabel="Predicted annual growth (%)",
    title="Locked test: observed vs predicted",
)
ax.set_aspect("equal", adjustable="box")
plt.show()

## 4. Environment-only SHAP explanation

### Cell purpose: compute SHAP once and isolate the environmental block

SHAP is first calculated for the complete pooled model, because that is the fitted predictor. This cell then retains only daytime noise, density, monoculture rate, sky-view factor, solar radiation, land-surface temperature, and nighttime illumination for reporting.

The summary records full and robust SHAP ranges, mean absolute SHAP, high-minus-low contrast, and feature–SHAP Spearman association.

**Output:** an environment-only SHAP summary table. Height, species, period, and site type are not shown in the following SHAP figures.

In [ ]:
shap_bundle = compute_environment_shap(trained, config)
display(shap_bundle["summary"].style.format(precision=4))

### SHAP beeswarm — what it means

Each dot is one locked-test tree-period observation. The x-axis is that environmental predictor's SHAP contribution to log-SGR: values right of zero raise the fitted prediction and values left of zero lower it. Colour represents the observed predictor value, from low to high. Features are ordered by mean absolute environmental SHAP.

The plot describes model associations, not causal effects.

**Cell output:** one environment-only beeswarm.

In [ ]:
from IPython.display import Image, display

beeswarm_path = plot_environment_beeswarm(shap_bundle)
display(Image(filename=str(beeswarm_path)))

### SHAP dependence — what it means

For each of the three leading environmental predictors, the x-axis is its observed value and the y-axis is its SHAP contribution. Curves, thresholds, or reversals reveal how the fitted response varies across the predictor range. Colour is restricted to the strongest approximate **environmental** interaction; height, species, period, and site type cannot appear on these colour bars.

**Cell output:** three environment-only dependence panels.

In [ ]:
dependence_path = plot_environment_dependence(shap_bundle)
display(Image(filename=str(dependence_path)))

### SHAP waterfall — what it means

This plot explains environmental contributions for one representative locked-test prediction. Its starting value already contains the expected model value plus the omitted height, species, period, and site-type contributions. The seven visible bars then show how the environment moves that contextual starting value to the complete prediction.

Red bars raise predicted log-SGR and blue bars lower it. This is an additive model decomposition, not a measured environmental offset.

**Cell output:** one environment-only waterfall.

In [ ]:
waterfall_path = plot_environment_waterfall(shap_bundle)
display(Image(filename=str(waterfall_path)))

### Spatial SHAP — what it means

This point map places the leading environmental predictor's SHAP contribution at sampled test-tree coordinates. Green points have positive contributions and red points have negative contributions to log-SGR. The map helps reveal local clustering in how the model uses that environmental predictor.

It is not an interpolated surface, a suitability map, or evidence of a causal spatial effect.

**Cell output:** one environment-only spatial SHAP point map and an audit CSV of mapped values.

In [ ]:
spatial_path = plot_environment_spatial(shap_bundle)
display(Image(filename=str(spatial_path)))

### Environment-only SHAP interpretation exercise

1. Which environmental predictor has the largest mean absolute SHAP?
2. Which dependence panel is monotonic, threshold-like, or nonlinear?
3. In the waterfall, do the combined environmental bars raise or lower the contextual prediction?
4. Where do positive and negative spatial contributions cluster?
5. What unmeasured local processes could produce similar spatial clustering?

Always describe these as fitted associations rather than causal environmental effects.

## 5. ONNX export and example test set

### Cell purpose: export and independently verify the model

This cell converts the fitted XGBoost trees to ONNX. The graph accepts the engineered feature matrix plus initial carbon stock, and returns log-SGR, specific growth rate, annual growth percentage, and annual kg C gain. ONNX Runtime must reproduce Python predictions within strict numerical tolerances.

**Output:** the ONNX model, feature schema, parity report, raw examples, engineered inputs, and expected predictions.

In [ ]:
onnx_outputs = export_onnx_and_examples(trained, config)
display(pd.Series(onnx_outputs["parity"], name="maximum absolute error").to_frame())
print("ONNX model:", onnx_outputs["onnx_path"])
print("Inputs:", onnx_outputs["schema"]["onnx_inputs"])
print("Outputs:", onnx_outputs["schema"]["onnx_outputs"])

display(pd.read_csv(OUTPUT_DIR / "examples" / "example_test_set_raw.csv").head())
display(pd.read_csv(OUTPUT_DIR / "examples" / "example_onnx_predictions.csv").head())

### Cell purpose: package all generated outputs

This cell creates a ZIP archive containing tables, figures, preprocessing artifacts, the XGBoost model, ONNX files, and examples. In Colab it starts a download.

**Output:** eCAADe_2026_outputs.zip.

In [ ]:
import shutil
archive_base = "/content/eCAADe_2026_outputs" if Path("/content").exists() else str(WORKSHOP_DIR / "eCAADe_2026_outputs")
archive = shutil.make_archive(archive_base, "zip", OUTPUT_DIR)
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download is available when this notebook runs in Google Colab.")

## Final reminder

This random tree-grouped test demonstrates the workflow. A planning-ready spatial diagnostic additionally needs spatially blocked deployment validation, reliability/domain masks, and local field verification. Do not attribute the full model R² to environmental predictors, and do not interpret SHAP as causal evidence.